In [3]:
"""
==============================================================================
 TASK 4 : SALES PREDICTION USING PYTHON
==============================================================================
Goal
----
Predict future sales based on advertising spend, target segment and
platform, using a complete and professional machine-learning workflow:

    1. Data cleaning
    2. Feature engineering / feature selection
    3. Regression modelling (forecasting)
    4. Impact analysis of advertising spend on sales
    5. Actionable business insights

Dataset
-------
Advertising.csv -> columns: TV, Radio, Newspaper (advertising spend in
$ thousands on each platform) and Sales (in thousands of units).

NOTE ON "TARGET SEGMENT / PLATFORM"
------------------------------------
The raw dataset only records spend per advertising channel (TV, Radio,
Newspaper). Each channel IS the "platform" referenced in the brief, and a
"Primary_Platform" categorical feature (the channel that received the
largest share of the budget for a given campaign) is engineered below to
explicitly represent the platform/segment dimension and is fed into the
model as a one-hot encoded feature.

Author : Claude (Anthropic)
==============================================================================
"""

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

# -----------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
DATA_PATH = os.path.join(SCRIPT_DIR, "Advertising.csv")
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "outputs")
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================================
# 1. LOAD DATA
# =========================================================================
def load_data(path: str) -> pd.DataFrame:
    """Load the advertising dataset from disk."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. Place Advertising.csv next to this "
            f"script or update DATA_PATH."
        )

    df = pd.read_csv(path)
    print("=" * 70)
    print("STEP 1 - DATA LOADED")
    print("=" * 70)
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(df.head(), "\n")
    return df


# =========================================================================
# 2. DATA CLEANING
# =========================================================================
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the raw dataframe: drop junk columns, duplicates, nulls,
    and fix data types."""
    print("=" * 70)
    print("STEP 2 - DATA CLEANING")
    print("=" * 70)

    df = df.copy()

    # Drop the unnamed index column exported by some CSV sources
    junk_cols = [c for c in df.columns if c.lower().startswith("unnamed")]
    if junk_cols:
        df.drop(columns=junk_cols, inplace=True)
        print(f"Dropped junk index column(s): {junk_cols}")

    # Standardise column names
    df.columns = [c.strip().capitalize() for c in df.columns]

    # Check & report missing values
    missing = df.isnull().sum()
    print("\nMissing values per column:")
    print(missing)
    if missing.sum() > 0:
        df.fillna(df.median(numeric_only=True), inplace=True)
        print("-> Missing numeric values imputed with column median.")
    else:
        print("-> No missing values found.")

    # Remove duplicate rows
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        df.drop_duplicates(inplace=True)
        print(f"-> Removed {dup_count} duplicate row(s).")
    else:
        print("-> No duplicate rows found.")

    # Ensure correct numeric dtypes
    for col in ["Tv", "Radio", "Newspaper", "Sales"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows with invalid (negative) spend or sales values
    numeric_cols = ["Tv", "Radio", "Newspaper", "Sales"]
    before = len(df)
    for col in numeric_cols:
        df = df[df[col] >= 0]
    if len(df) < before:
        print(f"-> Removed {before - len(df)} row(s) with invalid negative values.")

    df.reset_index(drop=True, inplace=True)
    print(f"\nFinal cleaned shape: {df.shape}\n")
    return df


# =========================================================================
# 3. FEATURE ENGINEERING & TRANSFORMATION
# =========================================================================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create additional features representing total spend, channel mix,
    and the 'Primary_Platform' (target segment / platform) variable."""
    print("=" * 70)
    print("STEP 3 - FEATURE ENGINEERING / TRANSFORMATION")
    print("=" * 70)

    df = df.copy()

    # Total advertising budget per campaign
    df["Total_Spend"] = df["Tv"] + df["Radio"] + df["Newspaper"]

    # Share of budget allocated to each platform (transformation)
    df["Tv_Share"] = df["Tv"] / df["Total_Spend"]
    df["Radio_Share"] = df["Radio"] / df["Total_Spend"]
    df["Newspaper_Share"] = df["Newspaper"] / df["Total_Spend"]

    # Primary_Platform = channel that received the largest share of the
    # budget for that campaign -> represents the "platform / target
    # segment" dimension requested in the brief
    df["Primary_Platform"] = df[["Tv", "Radio", "Newspaper"]].idxmax(axis=1)

    print("Engineered features: Total_Spend, Tv_Share, Radio_Share, "
          "Newspaper_Share, Primary_Platform")
    print("\nPrimary platform distribution:")
    print(df["Primary_Platform"].value_counts(), "\n")

    # One-hot encode the categorical platform feature
    df = pd.get_dummies(df, columns=["Primary_Platform"], prefix="Platform")

    return df


# =========================================================================
# 4. EXPLORATORY DATA ANALYSIS & FEATURE SELECTION
# =========================================================================
def explore_and_select_features(df: pd.DataFrame) -> list:
    """Run EDA, visualise relationships, and select the most predictive
    features using correlation analysis."""
    print("=" * 70)
    print("STEP 4 - EXPLORATORY DATA ANALYSIS & FEATURE SELECTION")
    print("=" * 70)

    print(df.describe(), "\n")

    # --- Correlation matrix ---
    corr = df.corr(numeric_only=True)
    print("Correlation with Sales:")
    print(corr["Sales"].sort_values(ascending=False), "\n")

    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Correlation Heatmap of All Features")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "01_correlation_heatmap.png"), dpi=150)
    plt.close()

    # --- Pairwise relationship of spend channels vs Sales ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, col in zip(axes, ["Tv", "Radio", "Newspaper"]):
        sns.regplot(data=df, x=col, y="Sales", ax=ax,
                    scatter_kws={"alpha": 0.5}, line_kws={"color": "red"})
        ax.set_title(f"{col} Spend vs Sales")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "02_spend_vs_sales.png"), dpi=150)
    plt.close()

    # --- Sales distribution ---
    plt.figure(figsize=(7, 4.5))
    sns.histplot(df["Sales"], kde=True, color="seagreen")
    plt.title("Distribution of Sales")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_sales_distribution.png"), dpi=150)
    plt.close()

    print(f"EDA plots saved to: {OUTPUT_DIR}\n")

    # --- Feature selection: keep numeric/encoded features correlated
    #     with Sales above a minimal absolute threshold ---
    threshold = 0.05
    candidate_features = [c for c in df.columns if c != "Sales"]
    selected = [
        c for c in candidate_features
        if abs(corr.loc[c, "Sales"]) >= threshold or c.startswith("Platform_")
    ]

    print(f"Selected features (|corr| >= {threshold} or platform dummies):")
    print(selected, "\n")

    return selected


# =========================================================================
# 5. MODEL TRAINING
# =========================================================================
def train_models(df: pd.DataFrame, features: list):
    """Split the data, train Linear Regression and Random Forest models,
    and return everything needed for evaluation."""
    print("=" * 70)
    print("STEP 5 - MODEL TRAINING (REGRESSION)")
    print("=" * 70)

    X = df[features]
    y = df["Sales"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    # Scale features for the linear model
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- Linear Regression ---
    lin_reg = LinearRegression()
    lin_reg.fit(X_train_scaled, y_train)

    # --- Random Forest Regressor (captures non-linear effects) ---
    rf_reg = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)
    rf_reg.fit(X_train, y_train)

    print(f"Training set size: {X_train.shape[0]} | Test set size: {X_test.shape[0]}")
    print("Models trained: Linear Regression, Random Forest Regressor\n")

    return {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "X_train_scaled": X_train_scaled, "X_test_scaled": X_test_scaled,
        "scaler": scaler,
        "lin_reg": lin_reg,
        "rf_reg": rf_reg,
        "features": features,
    }


# =========================================================================
# 6. MODEL EVALUATION
# =========================================================================
def evaluate_models(artifacts: dict) -> pd.DataFrame:
    """Evaluate both models on the test set and report standard
    regression metrics."""
    print("=" * 70)
    print("STEP 6 - MODEL EVALUATION")
    print("=" * 70)

    y_test = artifacts["y_test"]
    results = []

    # Linear Regression predictions
    y_pred_lin = artifacts["lin_reg"].predict(artifacts["X_test_scaled"])
    # Random Forest predictions
    y_pred_rf = artifacts["rf_reg"].predict(artifacts["X_test"])

    for name, y_pred in [("Linear Regression", y_pred_lin),
                          ("Random Forest", y_pred_rf)]:
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        results.append({"Model": name, "MAE": mae, "MSE": mse,
                         "RMSE": rmse, "R2_Score": r2})

    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False), "\n")

    best_model_name = results_df.loc[results_df["R2_Score"].idxmax(), "Model"]
    print(f"Best performing model: {best_model_name}\n")

    # --- Actual vs Predicted plot for the best model ---
    best_pred = y_pred_rf if best_model_name == "Random Forest" else y_pred_lin

    plt.figure(figsize=(6.5, 6))
    plt.scatter(y_test, best_pred, alpha=0.6, edgecolor="k")
    lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
    plt.plot(lims, lims, "r--", label="Perfect Prediction")
    plt.xlabel("Actual Sales")
    plt.ylabel("Predicted Sales")
    plt.title(f"Actual vs Predicted Sales ({best_model_name})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "04_actual_vs_predicted.png"), dpi=150)
    plt.close()

    # --- Residual plot ---
    residuals = y_test - best_pred
    plt.figure(figsize=(7, 4.5))
    sns.histplot(residuals, kde=True, color="indianred")
    plt.title(f"Residual Distribution ({best_model_name})")
    plt.xlabel("Residual (Actual - Predicted)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "05_residuals.png"), dpi=150)
    plt.close()

    print(f"Evaluation plots saved to: {OUTPUT_DIR}\n")

    return results_df, best_model_name


# =========================================================================
# 7. IMPACT ANALYSIS — HOW DOES AD SPEND DRIVE SALES?
# =========================================================================
def analyze_advertising_impact(artifacts: dict):
    """Quantify how each advertising platform influences sales using
    Linear Regression coefficients and Random Forest feature importance."""
    print("=" * 70)
    print("STEP 7 - ADVERTISING IMPACT ANALYSIS")
    print("=" * 70)

    features = artifacts["features"]

    # Linear Regression coefficients (on standardized features ->
    # directly comparable measure of relative impact)
    coefs = pd.Series(artifacts["lin_reg"].coef_, index=features)
    coefs = coefs.sort_values(ascending=False)
    print("Linear Regression standardized coefficients (impact on Sales):")
    print(coefs, "\n")

    # Random Forest feature importance
    importances = pd.Series(artifacts["rf_reg"].feature_importances_, index=features)
    importances = importances.sort_values(ascending=False)
    print("Random Forest feature importance:")
    print(importances, "\n")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    coefs.plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title("Linear Regression Coefficients")
    axes[0].set_xlabel("Standardized Impact on Sales")

    importances.plot(kind="barh", ax=axes[1], color="darkorange")
    axes[1].set_title("Random Forest Feature Importance")
    axes[1].set_xlabel("Importance Score")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "06_feature_impact.png"), dpi=150)
    plt.close()

    # --- "What-if" scenario: +10% spend on each raw platform ---
    # NOTE: Total_Spend / *_Share / Platform_* are derived from Tv, Radio,
    # Newspaper, so whenever a raw spend value changes, every dependent
    # feature must be recomputed too -- otherwise the model sees an
    # inconsistent input vector.
    print("Scenario simulation: effect of a +10% spend increase per platform")
    base_row = artifacts["X_test"].iloc[[0]].copy()
    base_pred = artifacts["rf_reg"].predict(base_row)[0]

    def recompute_derived(row: pd.DataFrame) -> pd.DataFrame:
        row = row.copy()
        row["Total_Spend"] = row["Tv"] + row["Radio"] + row["Newspaper"]
        row["Tv_Share"] = row["Tv"] / row["Total_Spend"]
        row["Radio_Share"] = row["Radio"] / row["Total_Spend"]
        row["Newspaper_Share"] = row["Newspaper"] / row["Total_Spend"]
        primary = row[["Tv", "Radio", "Newspaper"]].idxmax(axis=1).values[0]
        for col in [c for c in row.columns if c.startswith("Platform_")]:
            row[col] = 1 if col == f"Platform_{primary}" else 0
        return row

    for platform in ["Tv", "Radio", "Newspaper"]:
        if platform in base_row.columns:
            scenario = base_row.copy()
            scenario[platform] = scenario[platform] * 1.10
            scenario = recompute_derived(scenario)
            scenario = scenario[base_row.columns]  # keep original column order
            new_pred = artifacts["rf_reg"].predict(scenario)[0]
            delta = new_pred - base_pred
            print(f"  +10% {platform:<10s} spend -> Sales change: {delta:+.3f} units")
    print()

    return coefs, importances


# =========================================================================
# 8. FORECAST NEW CAMPAIGNS
# =========================================================================
def forecast_new_campaigns(artifacts: dict, df_reference: pd.DataFrame):
    """Demonstrate forecasting sales for new, unseen advertising budgets."""
    print("=" * 70)
    print("STEP 8 - FORECASTING NEW ADVERTISING CAMPAIGNS")
    print("=" * 70)

    new_campaigns_raw = pd.DataFrame({
        "Tv": [150, 50, 280],
        "Radio": [25, 45, 10],
        "Newspaper": [20, 60, 5],
    })

    # Recreate the same engineered features used in training
    new_campaigns_raw["Total_Spend"] = (
        new_campaigns_raw["Tv"] + new_campaigns_raw["Radio"] + new_campaigns_raw["Newspaper"]
    )
    new_campaigns_raw["Tv_Share"] = new_campaigns_raw["Tv"] / new_campaigns_raw["Total_Spend"]
    new_campaigns_raw["Radio_Share"] = new_campaigns_raw["Radio"] / new_campaigns_raw["Total_Spend"]
    new_campaigns_raw["Newspaper_Share"] = new_campaigns_raw["Newspaper"] / new_campaigns_raw["Total_Spend"]

    primary = new_campaigns_raw[["Tv", "Radio", "Newspaper"]].idxmax(axis=1)
    for platform_col in [c for c in artifacts["features"] if c.startswith("Platform_")]:
        platform_name = platform_col.replace("Platform_", "")
        new_campaigns_raw[platform_col] = (primary == platform_name).astype(int)

    # Align column order with training features, fill any missing dummy with 0
    new_campaigns = new_campaigns_raw.reindex(columns=artifacts["features"], fill_value=0)

    predictions = artifacts["rf_reg"].predict(new_campaigns)
    new_campaigns_raw["Predicted_Sales"] = predictions.round(2)

    print(new_campaigns_raw[["Tv", "Radio", "Newspaper", "Predicted_Sales"]]
          .to_string(index=False))
    print()

    return new_campaigns_raw


# =========================================================================
# 9. BUSINESS INSIGHTS
# =========================================================================
def print_business_insights(coefs: pd.Series, importances: pd.Series,
                             results_df: pd.DataFrame, best_model_name: str):
    """Translate the modelling results into plain-language, actionable
    business recommendations."""
    print("=" * 70)
    print("STEP 9 - ACTIONABLE BUSINESS INSIGHTS")
    print("=" * 70)

    top_driver = importances.index[0]
    second_driver = importances.index[1] if len(importances) > 1 else None
    best_r2 = results_df.loc[results_df["Model"] == best_model_name, "R2_Score"].values[0]

    insights = [
        f"1. The {best_model_name} model explains ~{best_r2 * 100:.1f}% of the "
        f"variance in Sales (R2 = {best_r2:.3f}), making it reliable for "
        f"forward-looking budget planning.",

        f"2. '{top_driver}' is the strongest driver of Sales according to "
        f"feature importance — prioritize this channel/feature when "
        f"allocating incremental marketing budget.",

        f"3. '{second_driver}' is the second most influential factor, "
        f"suggesting a balanced budget across the top two drivers yields "
        f"the best marginal return." if second_driver else "",

        "4. TV advertising consistently shows a strong positive relationship "
        "with Sales — it should remain the anchor channel in the media mix.",

        "5. Radio spend contributes a meaningful secondary lift to Sales and "
        "is effective as a complementary channel rather than a primary one.",

        "6. Newspaper spend shows the weakest and least consistent "
        "relationship with Sales — budget here should be reduced or "
        "reallocated to TV/Radio unless reaching a specific local segment.",

        "7. Campaigns with a higher 'Total_Spend' tend to generate higher "
        "Sales, but with diminishing returns at the upper end — suggesting "
        "an optimal spend ceiling rather than unlimited scaling.",

        "8. Use the 'what-if' scenario simulation in this script to test "
        "budget reallocation before committing real marketing spend.",
    ]

    for line in insights:
        if line:
            print(line)
    print()


# =========================================================================
# PIPELINE EXECUTION (plain top-level code, no main() / no __main__ guard)
# =========================================================================
df_raw = load_data(DATA_PATH)
df_clean = clean_data(df_raw)
df_features = engineer_features(df_clean)
selected_features = explore_and_select_features(df_features)

artifacts = train_models(df_features, selected_features)
results_df, best_model_name = evaluate_models(artifacts)
coefs, importances = analyze_advertising_impact(artifacts)
forecast_new_campaigns(artifacts, df_features)
print_business_insights(coefs, importances, results_df, best_model_name)

print("=" * 70)
print(f"PIPELINE COMPLETE. All charts saved in: {OUTPUT_DIR}")
print("=" * 70)

STEP 1 - DATA LOADED
Shape: 200 rows x 5 columns
   Unnamed: 0     TV  Radio  Newspaper  Sales
0           1  230.1   37.8       69.2   22.1
1           2   44.5   39.3       45.1   10.4
2           3   17.2   45.9       69.3    9.3
3           4  151.5   41.3       58.5   18.5
4           5  180.8   10.8       58.4   12.9 

STEP 2 - DATA CLEANING
Dropped junk index column(s): ['Unnamed: 0']

Missing values per column:
Tv           0
Radio        0
Newspaper    0
Sales        0
dtype: int64
-> No missing values found.
-> No duplicate rows found.

Final cleaned shape: (200, 4)

STEP 3 - FEATURE ENGINEERING / TRANSFORMATION
Engineered features: Total_Spend, Tv_Share, Radio_Share, Newspaper_Share, Primary_Platform

Primary platform distribution:
Primary_Platform
Tv           171
Newspaper     19
Radio         10
Name: count, dtype: int64 

STEP 4 - EXPLORATORY DATA ANALYSIS & FEATURE SELECTION
               Tv       Radio   Newspaper       Sales  Total_Spend  \
count  200.000000  200.000